# 02 — Data Cleaning

Loads the raw CSVs, runs data-quality checks (duplicates, missing values,
referential integrity, outliers, dtypes), fixes what's fixable, and writes
a single analysis-ready trip-level table to `data/processed/trips_analysis_ready.csv`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

riders = pd.read_csv(RAW_DIR / "riders.csv", parse_dates=["signup_date"])
drivers = pd.read_csv(RAW_DIR / "drivers.csv", parse_dates=["signup_date"])
vehicles = pd.read_csv(RAW_DIR / "vehicles.csv")
locations = pd.read_csv(RAW_DIR / "locations.csv")
trips = pd.read_csv(
    RAW_DIR / "trips.csv",
    parse_dates=["request_datetime", "pickup_datetime", "drop_datetime"],
)
payments = pd.read_csv(RAW_DIR / "payments.csv", parse_dates=["payment_datetime"])
ratings = pd.read_csv(RAW_DIR / "ratings.csv")

print(f"trips raw shape: {trips.shape}")

trips raw shape: (45025, 17)


## 1. Duplicate detection

In [2]:
n_dupes = trips.duplicated(subset="trip_id").sum()
print(f"Duplicate trip_id rows: {n_dupes}")
trips = trips.drop_duplicates(subset="trip_id", keep="first").reset_index(drop=True)
print(f"trips shape after de-dup: {trips.shape}")

Duplicate trip_id rows: 25
trips shape after de-dup: (45000, 17)


## 2. Missing-value analysis

In [3]:
missing_report = trips.isna().sum().to_frame("missing_count")
missing_report["missing_pct"] = (missing_report["missing_count"] / len(trips) * 100).round(2)
print(missing_report)

                     missing_count  missing_pct
trip_id                          0         0.00
rider_id                         0         0.00
driver_id                        0         0.00
vehicle_id                       0         0.00
pickup_location_id               0         0.00
drop_location_id                 0         0.00
request_datetime                 0         0.00
pickup_datetime               9493        21.10
drop_datetime                 9493        21.10
distance_km                      0         0.00
duration_min                  9849        21.89
base_fare                        0         0.00
surge_multiplier                 0         0.00
fare_amount                   9493        21.10
trip_status                      0         0.00
cancellation_reason          35507        78.90
payment_method                   0         0.00


`pickup_datetime`, `drop_datetime`, `fare_amount` are legitimately missing
for non-completed trips (the ride never started) — that's not a data
quality problem, it's the business reality of a cancelled ride. The one
real quality issue is `duration_min` missing on ~1% of *completed* trips,
which we impute from distance using the observed distance→duration
relationship (avoids dropping otherwise-valid completed trips).

In [4]:
completed_mask = trips.trip_status == "Completed"
valid_duration = trips.loc[completed_mask & trips.duration_min.notna()]
km_per_min = (valid_duration.distance_km / valid_duration.duration_min).median()
print(f"Median speed proxy: {1 / km_per_min:.2f} min per km")

missing_duration_mask = completed_mask & trips.duration_min.isna()
trips.loc[missing_duration_mask, "duration_min"] = (
    trips.loc[missing_duration_mask, "distance_km"] / km_per_min
).round(1)
print(f"Remaining missing duration_min on completed trips: "
      f"{trips.loc[completed_mask, 'duration_min'].isna().sum()}")

Median speed proxy: 2.98 min per km
Remaining missing duration_min on completed trips: 0


## 3. Referential integrity checks

In [5]:
orphan_riders = trips.loc[~trips.rider_id.isin(riders.rider_id)]
orphan_drivers = trips.loc[~trips.driver_id.isin(drivers.driver_id)]
orphan_pickup = trips.loc[~trips.pickup_location_id.isin(locations.location_id)]
orphan_drop = trips.loc[~trips.drop_location_id.isin(locations.location_id)]
print(f"Orphan rider refs: {len(orphan_riders)}, driver refs: {len(orphan_drivers)}, "
      f"pickup loc refs: {len(orphan_pickup)}, drop loc refs: {len(orphan_drop)}")
assert len(orphan_riders) == len(orphan_drivers) == len(orphan_pickup) == len(orphan_drop) == 0, \
    "Referential integrity violation found"

Orphan rider refs: 0, driver refs: 0, pickup loc refs: 0, drop loc refs: 0


## 4. Invalid-value checks

In [6]:
invalid_fare = trips.loc[completed_mask & (trips.fare_amount <= 0)]
invalid_distance = trips.loc[completed_mask & (trips.distance_km <= 0)]
invalid_surge = trips.loc[trips.surge_multiplier < 1.0]
print(f"Invalid fare: {len(invalid_fare)}, invalid distance: {len(invalid_distance)}, "
      f"invalid surge: {len(invalid_surge)}")

Invalid fare: 0, invalid distance: 0, invalid surge: 0


## 5. Outlier detection (distance, fare) via IQR

In [7]:
def iqr_bounds(series, k=3.0):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

dist_lo, dist_hi = iqr_bounds(trips.loc[completed_mask, "distance_km"])
fare_lo, fare_hi = iqr_bounds(trips.loc[completed_mask, "fare_amount"])
outlier_mask = completed_mask & (
    (trips.distance_km < dist_lo) | (trips.distance_km > dist_hi) |
    (trips.fare_amount < fare_lo) | (trips.fare_amount > fare_hi)
)
print(f"Outlier trips flagged (kept, not dropped — see `is_outlier` column): {outlier_mask.sum()}")
trips["is_outlier"] = outlier_mask

Outlier trips flagged (kept, not dropped — see `is_outlier` column): 340


## 6. Build the analysis-ready table

Join trips → vehicles, pickup/drop locations, riders, drivers, ratings and
add the calendar/derived columns EDA and statistics will use repeatedly.

In [8]:
trips_ready = (
    trips
    .merge(vehicles[["vehicle_id", "vehicle_type"]], on="vehicle_id", how="left")
    .merge(locations[["location_id", "city"]].rename(
        columns={"location_id": "pickup_location_id", "city": "pickup_city"}),
        on="pickup_location_id", how="left")
    .merge(locations[["location_id", "city"]].rename(
        columns={"location_id": "drop_location_id", "city": "drop_city"}),
        on="drop_location_id", how="left")
    .merge(riders[["rider_id", "signup_date", "age", "gender", "preferred_payment"]].rename(
        columns={"signup_date": "rider_signup_date", "gender": "rider_gender"}),
        on="rider_id", how="left")
    .merge(drivers[["driver_id", "signup_date", "avg_rating", "driver_status"]].rename(
        columns={"signup_date": "driver_signup_date", "avg_rating": "driver_avg_rating"}),
        on="driver_id", how="left")
    .merge(ratings[["trip_id", "rider_rating_for_driver", "driver_rating_for_rider"]],
        on="trip_id", how="left")
)

trips_ready["request_hour"] = trips_ready.request_datetime.dt.hour
trips_ready["request_day"] = trips_ready.request_datetime.dt.day_name()
trips_ready["request_dow"] = trips_ready.request_datetime.dt.dayofweek
trips_ready["request_month"] = trips_ready.request_datetime.dt.month
trips_ready["is_weekend"] = trips_ready.request_dow.isin([5, 6]).astype(int)
trips_ready["is_peak_hour"] = trips_ready.request_hour.isin([8, 9, 18, 19, 20]).astype(int)
trips_ready["is_cancelled"] = (trips_ready.trip_status != "Completed").astype(int)
trips_ready["fare_per_km"] = (trips_ready.fare_amount / trips_ready.distance_km.replace(0, np.nan)).round(2)
trips_ready["driver_experience_days"] = (
    trips_ready.request_datetime - trips_ready.driver_signup_date
).dt.days.clip(lower=0)
trips_ready["rider_tenure_days"] = (
    trips_ready.request_datetime - trips_ready.rider_signup_date
).dt.days.clip(lower=0)

print(trips_ready.shape)
trips_ready.head(3)

(45000, 40)


,trip_id,rider_id,driver_id,vehicle_id,pickup_location_id,drop_location_id,request_datetime,pickup_datetime,drop_datetime,distance_km,...,request_hour,request_day,request_dow,request_month,is_weekend,is_peak_hour,is_cancelled,fare_per_km,driver_experience_days,rider_tenure_days
0,1,1440,77,77,29,25,2025-01-01 00:20:00,2025-01-01 00:27:00,2025-01-01 01:38:30,24.20,...,0,Wednesday,2,1,0,0,0,29.23,280,0
1,2,231,252,252,3,7,2025-01-01 00:31:00,2025-01-01 00:40:00,2025-01-01 00:45:06,3.37,...,0,Wednesday,2,1,0,0,0,16.64,523,312
2,3,225,206,206,15,11,2025-01-01 02:14:00,NaT,NaT,9.16,...,2,Wednesday,2,1,0,0,1,NaN,254,327


## 7. Save

In [9]:
trips_ready.to_csv(PROCESSED_DIR / "trips_analysis_ready.csv", index=False)
print(f"Saved {len(trips_ready):,} rows -> data/processed/trips_analysis_ready.csv")

Saved 45,000 rows -> data/processed/trips_analysis_ready.csv
